# Corporate Sales Data Analysis
**Project:** Corporate Sales Analysis  
**Dataset:** `Corporate_sales_data.xlsx`

## Objective
Analyze sales performance, monthly trends, product performance, customer activity, and basic financial metrics using Python, pandas, and visualizations.

> **How to use:** Upload this notebook and `Corporate_sales_data.xlsx` to the same GitHub repository. Open the notebook in Google Colab or Jupyter and run the cells from top to bottom.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries imported successfully.")

## 2. Load the Excel Workbook
The workbook contains several sheets. This cell lists them and loads the main `Sales_data` sheet.

In [ ]:
file_path = Path("Corporate_sales_data.xlsx")
if not file_path.exists():
    candidates = list(Path(".").glob("*.xlsx"))
    if candidates:
        file_path = candidates[0]
    else:
        raise FileNotFoundError(
            "Please upload Corporate_sales_data.xlsx into the same folder as this notebook."
        )

excel = pd.ExcelFile(file_path)
print("Workbook:", file_path.name)
print("Sheets:", excel.sheet_names)

sales = pd.read_excel(file_path, sheet_name="Sales_data")
print("\nSales data shape:", sales.shape)
display(sales.head())

## 3. Inspect and Clean the Sales Data

In [ ]:
print("Column names:")
print(sales.columns.tolist())
print("\nData types and non-null counts:")
sales.info()
print("\nMissing values by column:")
display(sales.isna().sum().to_frame("Missing values"))
print("\nDuplicate rows:", sales.duplicated().sum())

# Standardize column labels and remove exact duplicate rows.
sales.columns = [str(c).strip() for c in sales.columns]
sales = sales.drop_duplicates().copy()

# Parse likely date columns safely.
for col in sales.columns:
    if "date" in col.lower():
        sales[col] = pd.to_datetime(sales[col], errors="coerce")

print("\nRows and columns after removing duplicates:", sales.shape)

## 4. Summary Statistics

In [ ]:
display(sales.describe(include="all").T)

## 5. Identify Important Sales Columns
This helper detects common column names so the analysis can work with slightly different spreadsheet headers.

In [ ]:
def find_col(frame, keywords):
    for col in frame.columns:
        name = str(col).lower().replace("_", " ").strip()
        if any(word in name for word in keywords):
            return col
    return None

date_col = find_col(sales, ["order date", "sales date", "date"])
sales_col = find_col(sales, ["sales", "revenue", "total amount", "amount"])
qty_col = find_col(sales, ["quantity", "qty", "units"])
product_col = find_col(sales, ["product name", "product"])
customer_col = find_col(sales, ["customer name", "customer"])
category_col = find_col(sales, ["category"])
region_col = find_col(sales, ["region", "state", "city"])

print("Detected columns")
print("Date:", date_col)
print("Sales/revenue:", sales_col)
print("Quantity:", qty_col)
print("Product:", product_col)
print("Customer:", customer_col)
print("Category:", category_col)
print("Region:", region_col)

if sales_col is None:
    print("\nNo obvious sales/revenue column was detected. Review the column names above and set sales_col manually.")
else:
    sales[sales_col] = pd.to_numeric(sales[sales_col], errors="coerce")
    print("\nTotal sales/revenue:", round(sales[sales_col].sum(), 2))
    print("Average sales value per row:", round(sales[sales_col].mean(), 2))
    print("Highest row sales value:", round(sales[sales_col].max(), 2))
    print("Lowest row sales value:", round(sales[sales_col].min(), 2))

## 6. Monthly Sales Trend

In [ ]:
if date_col and sales_col:
    monthly = (sales.dropna(subset=[date_col])
               .assign(Month=lambda d: d[date_col].dt.to_period("M").dt.to_timestamp())
               .groupby("Month", as_index=False)[sales_col].sum()
               .sort_values("Month"))
    display(monthly)
    if not monthly.empty:
        plt.figure()
        plt.plot(monthly["Month"], monthly[sales_col], marker="o")
        plt.title("Monthly Sales Trend")
        plt.xlabel("Month")
        plt.ylabel("Sales / Revenue")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("Monthly trend requires both a date column and a sales/revenue column. Check detected columns above.")

## 7. Product Performance

In [ ]:
if product_col and sales_col:
    product_sales = (sales.groupby(product_col, dropna=False)[sales_col]
                     .sum().sort_values(ascending=False).head(10).reset_index())
    display(product_sales)
    plt.figure()
    plt.bar(product_sales[product_col].astype(str), product_sales[sales_col])
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Product")
    plt.ylabel("Sales / Revenue")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Product performance requires product and sales/revenue columns.")

## 8. Category and Regional Performance

In [ ]:
for dimension, label in [(category_col, "Category"), (region_col, "Region")]:
    if dimension and sales_col:
        summary = (sales.groupby(dimension, dropna=False)[sales_col]
                   .sum().sort_values(ascending=False).reset_index())
        print(f"Sales by {label}:")
        display(summary)
    else:
        print(f"{label} analysis skipped because the required column was not detected.")

## 9. Customer Performance

In [ ]:
if customer_col and sales_col:
    customer_sales = (sales.groupby(customer_col, dropna=False)[sales_col]
                      .sum().sort_values(ascending=False).head(10).reset_index())
    print("Top 10 customers by sales:")
    display(customer_sales)
else:
    print("Customer analysis requires customer and sales/revenue columns.")

## 10. Quantity Analysis

In [ ]:
if qty_col:
    sales[qty_col] = pd.to_numeric(sales[qty_col], errors="coerce")
    print("Total quantity:", sales[qty_col].sum())
    print("Average quantity per row:", round(sales[qty_col].mean(), 2))
    if product_col:
        quantity_by_product = (sales.groupby(product_col, dropna=False)[qty_col]
                               .sum().sort_values(ascending=False).head(10).reset_index())
        display(quantity_by_product)
else:
    print("Quantity analysis skipped because a quantity column was not detected.")

## 11. Findings and Conclusion
Review the tables and charts generated above, then complete the findings below with values from your output.

- **Total sales/revenue:** Fill in from Section 5.
- **Monthly trend:** Describe the strongest and weakest months from Section 6.
- **Top product:** Identify the highest-selling product from Section 7.
- **Category/region:** Mention the leading category or region if available.
- **Customer insight:** Identify the highest-value customer if available.

### Conclusion
This notebook provides a reproducible starting point for corporate sales analysis. The final interpretation should be based on the actual tables and charts produced from the workbook.

## 12. Export Cleaned Data (Optional)

In [ ]:
cleaned_file = Path("cleaned_sales_data.csv")
sales.to_csv(cleaned_file, index=False)
print("Cleaned data saved to:", cleaned_file)